# SimpleNews S0–S5

## 0. Setup and shared-module import

This cell:
1. makes sure a local `src/` package exists (using the shared repo files if they are present beside the notebook),
2. adds the repo root to `sys.path`,
3. imports the shared modules.

In [1]:

from pathlib import Path
import sys
import os
import shutil

ROOT = Path('.').resolve()
SRC = ROOT / "src"

# Always refresh a lightweight src/ package from the current root module files.
# This lets the notebook use the improved drop-in modules without embedding them in cells.
SRC.mkdir(parents=True, exist_ok=True)

module_files = [
    "__init__.py",
    "config.py",
    "dataset_bootstrap.py",
    "evaluation.py",
    "io_utils.py",
    "pipeline.py",
    "preprocess.py",
    "selectors.py",
    "simplify.py",
]

for name in module_files:
    src_file = ROOT / name
    dst_file = SRC / name
    if not src_file.exists():
        continue
    if dst_file.exists() or dst_file.is_symlink():
        try:
            dst_file.unlink()
        except Exception:
            pass
    try:
        os.symlink(src_file, dst_file)
        print(f"Linked {name} -> {src_file.name}")
    except Exception:
        shutil.copy2(src_file, dst_file)
        print(f"Copied {name} -> src/{name}")

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.config import DEFAULT_CONFIG, Paths
from src.dataset_bootstrap import ensure_dataset
from src.io_utils import ensure_dirs, load_csv, standardize_dataframe
from src.pipeline import generate_system_outputs
from src.preprocess import (
    split_sentences,
    tokenize,
    article_features,
    extract_entities,
    extract_numbers,
    extract_dates,
    token_frequency_rank,
    normalize_whitespace,
)
from src.simplify import conservative_lexical_simplify, protected_spans
from src.evaluation import evaluate_output

print("ROOT:", ROOT)
print("SRC exists:", SRC.exists(), "->", SRC)


ROOT: /Users/gm/Desktop/2
SRC exists: True -> /Users/gm/Desktop/2/src


## 1. Install / import extra packages for S4 / S5

In [2]:
import sys, subprocess, importlib.util

REQUIRED = {
    "pandas": "pandas",
    "numpy": "numpy",
    "torch": "torch",
    "transformers": "transformers",
    "datasets": "datasets",
}

missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)

In [3]:
import gc
import math
import copy
import warnings
import time
from typing import Dict, List, Optional

import pandas as pd
import numpy as np
import torch
from IPython.display import HTML, display
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers.utils import logging as hf_logging

# Silence repetitive generation/config warnings that otherwise flood notebook output
hf_logging.set_verbosity_error()
warnings.filterwarnings("ignore", message=r".*max_new_tokens.*max_length.*")
warnings.filterwarnings("ignore", message=r".*min_new_tokens.*min_length.*")
warnings.filterwarnings("ignore", message=r".*forced_bos_token_id.*")
warnings.filterwarnings("ignore", message=r".*tie shared.weight to lm_head.weight.*")

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

def runtime_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return "mps"
    return "cpu"

DEVICE = runtime_device()
print("Selected device:", DEVICE)

class NotebookProgress:
    def __init__(self, total: int, desc: str, enabled: bool = True, update_every: int = 5):
        self.total = max(int(total), 1)
        self.desc = desc
        self.enabled = enabled
        self.update_every = max(1, int(update_every))
        self.start = time.time()
        self.last_doc = None
        self.handle = None
        if self.enabled:
            self.handle = display(self._render(0), display_id=True)

    def _render(self, n: int):
        pct = min(100.0, 100.0 * n / self.total)
        elapsed = time.time() - self.start
        rate = n / elapsed if elapsed > 0 else 0.0
        eta = (self.total - n) / rate if rate > 0 else 0.0
        elapsed_txt = f"{elapsed/60:.1f} min"
        eta_txt = "--" if rate <= 0 else f"{eta/60:.1f} min"
        last_txt = "" if self.last_doc is None else f" | last_doc={self.last_doc}"
        return HTML(f'''
        <div style="font-family:Arial,sans-serif; margin:8px 0;">
          <div style="margin-bottom:4px;"><b>{self.desc}</b>: {n}/{self.total} ({pct:.1f}%) | elapsed {elapsed_txt} | ETA {eta_txt}{last_txt}</div>
          <div style="width:100%; background:#e9ecef; border-radius:8px; overflow:hidden; height:14px;">
            <div style="width:{pct:.1f}%; background:#2f80ed; height:14px;"></div>
          </div>
        </div>
        ''')

    def update(self, n: int, last_doc: Optional[str] = None, force: bool = False):
        if not self.enabled or self.handle is None:
            return
        if last_doc is not None:
            self.last_doc = last_doc
        if force or n == self.total or n % self.update_every == 0:
            self.handle.update(self._render(n))

    def close(self, last_doc: Optional[str] = None):
        self.update(self.total, last_doc=last_doc, force=True)

def make_progress_bar(total: int, desc: str, enabled: bool = True, update_every: int = 5):
    if not enabled:
        return None
    return NotebookProgress(total=total, desc=desc, enabled=True, update_every=update_every)


Selected device: mps


## 2. Configuration

In [4]:

paths = Paths(
    root=ROOT,
    data_dir=ROOT / "data",
    output_dir=ROOT / "outputs",
    figure_dir=ROOT / "outputs/figures",
    table_dir=ROOT / "outputs/tables",
    sample_dir=ROOT / "outputs/samples",
)
ensure_dirs(paths.data_dir, paths.output_dir, paths.figure_dir, paths.table_dir, paths.sample_dir)

# Tune the improved shared pipeline a bit more aggressively than the default lightweight config.
DEFAULT_CONFIG.selector.max_sentences = 3
DEFAULT_CONFIG.selector.similarity_threshold = 0.08
DEFAULT_CONFIG.selector.top_entity_k = 8
DEFAULT_CONFIG.selector.require_number_coverage = True
DEFAULT_CONFIG.selector.require_date_coverage = True
DEFAULT_CONFIG.simplifier.enabled = True
DEFAULT_CONFIG.simplifier.max_replacements_per_doc = 10
DEFAULT_CONFIG.simplifier.use_glossary_fallback = True

# Dataset split to run on
RUN_SPLIT = "validation"   # "train", "validation", or "test"
RUN_LIMIT = None           # set to e.g. 100 for debugging, None for full split

# LLM systems
ENABLE_LLM_SYSTEMS = True

# Model choices
S4_MODEL_NAME = "google/flan-t5-base"         # controlled rewrite
S5_MODEL_NAME = "facebook/bart-large-cnn"     # full-article summarization baseline

# Generation lengths / budgets
S4_MAX_NEW_TOKENS = 144
S5_MAX_NEW_TOKENS = 128
S5_MIN_NEW_TOKENS = 56
S5_CHUNK_WORD_BUDGET = 380

# S4 guard: loosened a bit so the rewrite is allowed to change more than before.
S4_MIN_FACT_RECALL = 0.70

# Save names
RUN_TAG = f"{RUN_SPLIT}_s0_s5_repo_integrated_v18"

print("RUN_SPLIT =", RUN_SPLIT)
print("RUN_LIMIT =", RUN_LIMIT)
print("ENABLE_LLM_SYSTEMS =", ENABLE_LLM_SYSTEMS)
print("S4_MODEL_NAME =", S4_MODEL_NAME)
print("S5_MODEL_NAME =", S5_MODEL_NAME)
print("DEFAULT_CONFIG =", DEFAULT_CONFIG)


RUN_SPLIT = validation
RUN_LIMIT = None
ENABLE_LLM_SYSTEMS = True
S4_MODEL_NAME = google/flan-t5-base
S5_MODEL_NAME = facebook/bart-large-cnn
DEFAULT_CONFIG = ExperimentConfig(random_seed=42, sample_size=200, systems=['s0_lead3', 's1_textrank', 's2_coverage', 's3_simplified'], selector=SelectorConfig(max_sentences=3, similarity_threshold=0.08, top_entity_k=8, require_number_coverage=True, require_date_coverage=True), simplifier=SimplifyConfig(enabled=True, max_word_length=9, min_word_frequency_rank=3000, max_replacements_per_doc=10, use_glossary_fallback=True, protected_words=['said', 'says', 'mr', 'mrs', 'ms', 'government', 'minister', 'president', 'police', 'court', 'judge', 'company', 'officials']))


## 3. Dataset bootstrap + load selected split

In [5]:
dataset_status = ensure_dataset(
    data_dir=paths.data_dir,
    auto_download=True,
    force_download=False,
)
print(dataset_status)

split_path = paths.data_dir / f"{RUN_SPLIT}.csv"
raw_df, colmap = load_csv(split_path)
df = standardize_dataframe(raw_df, colmap)

# Keep doc_id as the repo-native identifier.
# Also expose an id alias for compatibility if someone downstream expects it.
if "doc_id" not in df.columns:
    raise ValueError("Expected repo-standardized dataframe to contain 'doc_id'.")
df["doc_id"] = df["doc_id"].astype(str)
df["id"] = df["doc_id"]

if RUN_LIMIT is not None:
    df = df.iloc[:RUN_LIMIT].copy()

print("Loaded split:", split_path)
print("Rows:", len(df))
df.head(3)


Downloaded CNN/DailyMail and saved local CSV files.
Loaded split: /Users/gm/Desktop/2/data/validation.csv
Rows: 13368


,doc_id,title,article,reference,id
0,0,,"(CNN)Share, and your gift will be multiplied. ...",Zully Broussard decided to give a kidney to a ...,0
1,1,,"(CNN)On the 6th of April 1996, San Jose Clash ...",The 20th MLS season begins this weekend .\nLea...,1
2,2,,"(CNN)French striker Bafetimbi Gomis, who has a...",Bafetimbi Gomis collapses within 10 minutes of...,2


## 4. Shared I/O helpers

In [6]:
def normalize_pipeline_input(df_in: pd.DataFrame) -> pd.DataFrame:
    out = df_in.copy()
    if "doc_id" not in out.columns and "id" in out.columns:
        out["doc_id"] = out["id"]
    required = ["doc_id", "article", "reference"]
    for col in required:
        if col not in out.columns:
            raise ValueError(f"Missing required input column: {col}")
    out["doc_id"] = out["doc_id"].astype(str)
    out["id"] = out["doc_id"]  # compatibility alias
    out["article"] = out["article"].fillna("").astype(str)
    out["reference"] = out["reference"].fillna("").astype(str)
    if "title" not in out.columns:
        out["title"] = ""
    return out

def build_pipeline_output_row(
    *,
    row_id: str,
    system: str,
    article: str,
    reference: str,
    output: str,
    extra: Optional[Dict[str, object]] = None,
) -> Dict[str, object]:
    base = {
        "doc_id": row_id,
        "id": row_id,  # compatibility alias
        "system": system,
        "article": article,
        "reference": reference,
        "output": output,
    }
    if extra:
        base.update(extra)
    return base

def append_shared_metrics(out_df: pd.DataFrame) -> pd.DataFrame:
    metrics_rows = []
    for row in out_df.itertuples(index=False):
        metrics = evaluate_output(source=row.article, reference=row.reference, pred=row.output)
        metrics_rows.append(metrics)
    metrics_df = pd.DataFrame(metrics_rows)
    return pd.concat([out_df.reset_index(drop=True), metrics_df.reset_index(drop=True)], axis=1)


## 5. Shared S0–S3 using the repo pipeline

In [7]:
SYSTEM_MAP = {
    "s0_lead3": "S0",
    "s1_textrank": "S1",
    "s2_coverage": "S2",
    "s3_simplified": "S3",
}

def run_shared_s0_s3(df_in: pd.DataFrame) -> pd.DataFrame:
    # Use the shared repo pipeline AS-IS and keep its computed metrics.
    work = df_in.copy()
    shared_df = generate_system_outputs(work[["doc_id", "title", "article", "reference"]], DEFAULT_CONFIG).copy()
    shared_df["doc_id"] = shared_df["doc_id"].astype(str)
    shared_df["system"] = shared_df["system"].map(SYSTEM_MAP)

    # Add the teammate-facing `id` alias without removing repo-native `doc_id`.
    insert_at = list(shared_df.columns).index("doc_id") + 1
    shared_df.insert(insert_at, "id", shared_df["doc_id"])
    return shared_df

df_in = normalize_pipeline_input(df)
shared_outputs = run_shared_s0_s3(df_in)

# Reuse the shared S2 outputs directly for S4 instead of recomputing coverage-aware TextRank.
S2_LOOKUP = (
    shared_outputs.loc[shared_outputs["system"] == "S2", ["doc_id", "output"]]
    .drop_duplicates(subset=["doc_id"])
    .set_index("doc_id")["output"]
    .to_dict()
)

print("Shared outputs rows:", len(shared_outputs))
print("Unique docs in S2 lookup:", len(S2_LOOKUP))
shared_outputs.head(8)

Shared outputs rows: 53472
Unique docs in S2 lookup: 13368


,doc_id,id,system,system_label,title,article,reference,output,replacements,glossary,...,entity_unsupported_ratio,number_precision,number_f1,number_unsupported_ratio,date_precision,date_f1,date_unsupported_ratio,novel_token_ratio,copy_token_ratio,novel_content_token_ratio
0,0,0,S0,S0 Lead-3,,"(CNN)Share, and your gift will be multiplied. ...",Zully Broussard decided to give a kidney to a ...,"(CNN)Share, and your gift will be multiplied. ...",[],[],...,0.0,0.0,0.000000,0.0,1.0,1.0,0.0,0.0,1.0,0.0
1,0,0,S1,S1 TextRank,,"(CNN)Share, and your gift will be multiplied. ...",Zully Broussard decided to give a kidney to a ...,"Here's how the super swap works, according to ...",[],[],...,0.0,0.0,0.000000,0.0,1.0,1.0,0.0,0.0,1.0,0.0
2,0,0,S2,S2 Coverage-aware TextRank,,"(CNN)Share, and your gift will be multiplied. ...",Zully Broussard decided to give a kidney to a ...,But maybe your kidney is a match for his siste...,[],[],...,0.0,0.0,0.000000,0.0,1.0,1.0,0.0,0.0,1.0,0.0
3,0,0,S3,S3 Coverage-aware + simplification,,"(CNN)Share, and your gift will be multiplied. ...",Zully Broussard decided to give a kidney to a ...,But maybe your kidney is a match for his siste...,[],[],...,0.0,0.0,0.000000,0.0,1.0,1.0,0.0,0.0,1.0,0.0
4,1,1,S0,S0 Lead-3,,"(CNN)On the 6th of April 1996, San Jose Clash ...",The 20th MLS season begins this weekend .\nLea...,"(CNN)On the 6th of April 1996, San Jose Clash ...",[],[],...,0.0,1.0,0.181818,0.0,1.0,0.2,0.0,0.0,1.0,0.0
5,1,1,S1,S1 TextRank,,"(CNN)On the 6th of April 1996, San Jose Clash ...",The 20th MLS season begins this weekend .\nLea...,"This weekend 60,000 fans are expected to witne...",[],[],...,0.0,1.0,0.095238,0.0,0.0,0.0,0.0,0.0,1.0,0.0
6,1,1,S2,S2 Coverage-aware TextRank,,"(CNN)On the 6th of April 1996, San Jose Clash ...",The 20th MLS season begins this weekend .\nLea...,World Cup winners Kaka and David Villa will tu...,[],[],...,0.0,1.0,0.095238,0.0,1.0,0.2,0.0,0.0,1.0,0.0
7,1,1,S3,S3 Coverage-aware + simplification,,"(CNN)On the 6th of April 1996, San Jose Clash ...",The 20th MLS season begins this weekend .\nLea...,World Cup winners Kaka and David Villa will tu...,[],[],...,0.0,1.0,0.095238,0.0,1.0,0.2,0.0,0.0,1.0,0.0


## 6. S4 / S5 model helpers

In [8]:
_model_cache = {}

def clear_model_cache():
    global _model_cache
    _model_cache = {}
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE == "mps":
        try:
            torch.mps.empty_cache()
        except Exception:
            pass

def _clean_generation_config(model, tokenizer):
    gen_cfg = copy.deepcopy(model.generation_config)
    # Remove conflicting defaults that spam warnings when max_new_tokens/min_new_tokens are passed.
    if hasattr(gen_cfg, "max_length"):
        gen_cfg.max_length = None
    if hasattr(gen_cfg, "min_length"):
        gen_cfg.min_length = None
    # Some seq2seq checkpoints expect a BOS id in generation config.
    if getattr(gen_cfg, "forced_bos_token_id", None) is None and getattr(tokenizer, "bos_token_id", None) is not None:
        gen_cfg.forced_bos_token_id = tokenizer.bos_token_id
    return gen_cfg

def load_model_bundle(model_name: str):
    if model_name in _model_cache:
        return _model_cache[model_name]
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    if DEVICE != "cpu":
        model = model.to(DEVICE)
    model.eval()
    model.generation_config = _clean_generation_config(model, tokenizer)
    _model_cache[model_name] = (tokenizer, model)
    return _model_cache[model_name]

def generate_with_model(model_name: str, prompt: str, max_new_tokens: int, min_new_tokens: int = 0) -> str:
    tokenizer, model = load_model_bundle(model_name)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            generation_config=model.generation_config,
            max_new_tokens=max_new_tokens,
            min_new_tokens=min_new_tokens,
            num_beams=4,
            do_sample=False,
            early_stopping=True,
            pad_token_id=(tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id),
        )
    text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return normalize_whitespace(text)

def fact_recall_against_source(source_text: str, candidate_text: str) -> float:
    source_items = set()
    source_items.update([x.lower() for x in extract_entities(source_text)])
    source_items.update([x.lower() for x in extract_numbers(source_text)])
    source_items.update([x.lower() for x in extract_dates(source_text)])
    if not source_items:
        return 1.0
    cand_items = set()
    cand_items.update([x.lower() for x in extract_entities(candidate_text)])
    cand_items.update([x.lower() for x in extract_numbers(candidate_text)])
    cand_items.update([x.lower() for x in extract_dates(candidate_text)])
    return len(source_items & cand_items) / len(source_items)


## 7. S4 = S2 + constrained LLM rewrite

In [9]:

def build_s4_prompt(selected_text: str) -> str:
    return (
        "Rewrite the following news summary into simpler English for adult ESL readers.\n"
        "Rules:\n"
        "- Preserve all names, dates, numbers, money amounts, and percentages exactly.\n"
        "- Do not add any new facts.\n"
        "- Use 3 to 5 short, clear sentences.\n"
        "- Aim for about 60 to 90 words.\n"
        "- Prefer simple vocabulary, but keep the meaning faithful.\n\n"
        f"Summary:\n{selected_text}\n\n"
        "Simplified summary:"
    )

def get_s2_output_for_row(row: pd.Series) -> str:
    row_id = str(row["doc_id"])
    if row_id in S2_LOOKUP:
        return S2_LOOKUP[row_id]

    # Fallback only if the lookup is missing; normally this path should not run.
    title = row["title"] if "title" in row else ""
    return __import__("src.selectors", fromlist=["coverage_aware_select"]).coverage_aware_select(
        row["article"],
        title=title,
        max_sentences=DEFAULT_CONFIG.selector.max_sentences,
        top_entity_k=DEFAULT_CONFIG.selector.top_entity_k,
        similarity_threshold=DEFAULT_CONFIG.selector.similarity_threshold,
        require_number_coverage=DEFAULT_CONFIG.selector.require_number_coverage,
        require_date_coverage=DEFAULT_CONFIG.selector.require_date_coverage,
    )

def run_s4_on_example(row: pd.Series) -> Dict[str, object]:
    s2_text = get_s2_output_for_row(row)
    prompt = build_s4_prompt(s2_text)
    raw = generate_with_model(S4_MODEL_NAME, prompt, max_new_tokens=S4_MAX_NEW_TOKENS, min_new_tokens=48)
    guard = fact_recall_against_source(s2_text, raw)
    if guard < S4_MIN_FACT_RECALL:
        final = s2_text
        fallback_used = True
    else:
        final = raw
        fallback_used = False
    return {
        "output": final,
        "s4_raw": raw,
        "s4_guard_fact_recall": guard,
        "s4_fallback_used": fallback_used,
    }


## 8. S5 = full-article LLM baseline

In [10]:

def chunk_article_by_words(article: str, word_budget: int = S5_CHUNK_WORD_BUDGET) -> List[str]:
    sents = split_sentences(article)
    chunks, cur, cur_words = [], [], 0
    for sent in sents:
        n = len(tokenize(sent))
        if cur and cur_words + n > word_budget:
            chunks.append(" ".join(cur).strip())
            cur, cur_words = [sent], n
        else:
            cur.append(sent)
            cur_words += n
    if cur:
        chunks.append(" ".join(cur).strip())
    return chunks

def build_s5_prompt(article_text: str) -> str:
    return (
        "Summarize this news article in plain English for adult ESL readers.\n"
        "Rules:\n"
        "- Keep the most important names, dates, numbers, and places.\n"
        "- Do not add any new facts.\n"
        "- Write 3 to 5 short, coherent sentences.\n"
        "- Aim for about 60 to 90 words.\n\n"
        f"Article:\n{article_text}\n\n"
        "Summary:"
    )

def run_s5_on_example(row: pd.Series) -> Dict[str, object]:
    article = row["article"]
    if len(tokenize(article)) <= S5_CHUNK_WORD_BUDGET:
        out = generate_with_model(
            S5_MODEL_NAME,
            build_s5_prompt(article),
            max_new_tokens=S5_MAX_NEW_TOKENS,
            min_new_tokens=S5_MIN_NEW_TOKENS,
        )
        return {"output": out, "s5_mode": "single_pass"}

    chunks = chunk_article_by_words(article)
    chunk_summaries = []
    for chunk in chunks:
        chunk_out = generate_with_model(
            S5_MODEL_NAME,
            build_s5_prompt(chunk),
            max_new_tokens=max(64, S5_MAX_NEW_TOKENS // 2),
            min_new_tokens=24,
        )
        chunk_summaries.append(chunk_out)

    merged_source = " ".join(chunk_summaries)
    final = generate_with_model(
        S5_MODEL_NAME,
        build_s5_prompt(merged_source),
        max_new_tokens=S5_MAX_NEW_TOKENS,
        min_new_tokens=S5_MIN_NEW_TOKENS,
    )
    return {"output": final, "s5_mode": "chunk_then_merge", "s5_num_chunks": len(chunks)}


## 9. Run S4 / S5 in the shared row format (separate loops for clearer progress and lower memory pressure)

In [11]:
def run_s4_system(df_in: pd.DataFrame, show_progress: bool = True, heartbeat_every: int = 25) -> pd.DataFrame:
    rows = []
    total = len(df_in)
    print(f"Running S4 on {total} examples...", flush=True)
    pbar = make_progress_bar(total=total, desc="Running S4", enabled=show_progress, update_every=5)
    last_doc = None

    try:
        for idx, row in enumerate(df_in.itertuples(index=False), start=1):
            row_series = pd.Series(row._asdict())
            row_id = str(row_series["doc_id"])
            last_doc = row_id

            s4_payload = run_s4_on_example(row_series)
            rows.append(build_pipeline_output_row(
                row_id=row_id,
                system="S4",
                article=row_series["article"],
                reference=row_series["reference"],
                output=s4_payload["output"],
                extra={k: v for k, v in s4_payload.items() if k != "output"},
            ))

            if pbar is not None:
                pbar.update(idx, last_doc=row_id)

    finally:
        if pbar is not None:
            pbar.close(last_doc=last_doc)

    out = pd.DataFrame(rows)
    print("Scoring S4 outputs...", flush=True)
    scored = append_shared_metrics(out)
    print("Finished S4.", flush=True)
    return scored

def run_s5_system(df_in: pd.DataFrame, show_progress: bool = True, heartbeat_every: int = 10) -> pd.DataFrame:
    rows = []
    total = len(df_in)
    print(f"Running S5 on {total} examples...", flush=True)
    pbar = make_progress_bar(total=total, desc="Running S5", enabled=show_progress, update_every=2)
    last_doc = None

    try:
        for idx, row in enumerate(df_in.itertuples(index=False), start=1):
            row_series = pd.Series(row._asdict())
            row_id = str(row_series["doc_id"])
            last_doc = row_id

            s5_payload = run_s5_on_example(row_series)
            rows.append(build_pipeline_output_row(
                row_id=row_id,
                system="S5",
                article=row_series["article"],
                reference=row_series["reference"],
                output=s5_payload["output"],
                extra={k: v for k, v in s5_payload.items() if k != "output"},
            ))

            if pbar is not None:
                pbar.update(idx, last_doc=row_id)

    finally:
        if pbar is not None:
            pbar.close(last_doc=last_doc)

    out = pd.DataFrame(rows)
    print("Scoring S5 outputs...", flush=True)
    scored = append_shared_metrics(out)
    print("Finished S5.", flush=True)
    return scored

if ENABLE_LLM_SYSTEMS:
    s4_outputs = run_s4_system(df_in, show_progress=True, heartbeat_every=25)

    # Free FLAN-T5 before loading BART.
    clear_model_cache()

    s5_outputs = run_s5_system(df_in, show_progress=True, heartbeat_every=10)

    llm_outputs = pd.concat([s4_outputs, s5_outputs], ignore_index=True, sort=False)
else:
    llm_outputs = pd.DataFrame(columns=["doc_id", "id", "system", "article", "reference", "output"])

print("LLM output rows:", len(llm_outputs))
llm_outputs.head(6)


Running S4 on 13368 examples...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

Scoring S4 outputs...
Finished S4.
Running S5 on 13368 examples...


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Scoring S5 outputs...
Finished S5.
LLM output rows: 26736


,doc_id,id,system,article,reference,output,s4_raw,s4_guard_fact_recall,s4_fallback_used,rouge1,...,number_f1,number_unsupported_ratio,date_precision,date_f1,date_unsupported_ratio,novel_token_ratio,copy_token_ratio,novel_content_token_ratio,s5_mode,s5_num_chunks
0,0,0,S4,"(CNN)Share, and your gift will be multiplied. ...",Zully Broussard decided to give a kidney to a ...,But maybe your kidney is a match for his siste...,But maybe your kidney is a match for his siste...,1.000000,False,0.159091,...,0.000000,0.0,1.0,1.0,0.0,0.0,1.0,0.0,NaN,NaN
1,1,1,S4,"(CNN)On the 6th of April 1996, San Jose Clash ...",The 20th MLS season begins this weekend .\nLea...,World Cup winners Kaka and David Villa will tu...,We have a salary cap. We have a salary cap. Ei...,0.625000,True,0.166667,...,0.095238,0.0,1.0,0.2,0.0,0.0,1.0,0.0,NaN,NaN
2,2,2,S4,"(CNN)French striker Bafetimbi Gomis, who has a...",Bafetimbi Gomis collapses within 10 minutes of...,"(CNN)French striker Bafetimbi Gomis, who has a...","Bafetimbi Gomis, who has a history of fainting...",0.416667,True,0.368000,...,0.666667,0.0,0.0,0.0,0.0,0.0,1.0,0.0,NaN,NaN
3,3,3,S4,(CNN)It was an act of frustration perhaps more...,Rory McIlroy throws club into water at WGC Cad...,McIlroy composed himself to finish with a seco...,Rory McIlroy finished with a second round of 7...,0.300000,True,0.038835,...,0.727273,0.0,1.0,1.0,0.0,0.0,1.0,0.0,NaN,NaN
4,4,4,S4,(CNN)A Pennsylvania community is pulling toget...,"Cayman Naib, 13, hasn't been heard from since ...","""We have spoken to his friends and they do not...","""We have spoken to his friends and they do not...",1.000000,False,0.111111,...,0.333333,0.0,1.0,1.0,0.0,0.0,1.0,0.0,NaN,NaN
5,5,5,S4,(CNN)My vote for Father of the Year goes to Cu...,Ruben Navarrette: Schilling deserves praise fo...,"The drama started, innocently enough, on Febru...","The drama started, innocently enough, on Febru...",1.000000,False,0.160000,...,0.666667,0.0,1.0,1.0,0.0,0.0,1.0,0.0,NaN,NaN


## 10. Combine all systems and save shared-format outputs

In [12]:
combined = pd.concat([shared_outputs, llm_outputs], ignore_index=True, sort=False)
# keep core columns first
core_cols = ["doc_id", "id", "system", "article", "reference", "output"]
other_cols = [c for c in combined.columns if c not in core_cols]
combined = combined[core_cols + other_cols]

combined_path = paths.output_dir / f"{RUN_TAG}_all_system_outputs.csv"
combined.to_csv(combined_path, index=False)

# Save one CSV per system too
per_system_dir = paths.output_dir / "per_system"
ensure_dirs(per_system_dir)
for sys_name, sub in combined.groupby("system"):
    sub.to_csv(per_system_dir / f"{RUN_TAG}_{sys_name}.csv", index=False)

print("Saved combined outputs to:", combined_path.resolve())
print("Saved per-system CSVs to:", per_system_dir.resolve())
combined.head(12)


Saved combined outputs to: /Users/gm/Desktop/2/outputs/validation_s0_s5_repo_integrated_v18_all_system_outputs.csv
Saved per-system CSVs to: /Users/gm/Desktop/2/outputs/per_system


,doc_id,id,system,article,reference,output,system_label,title,replacements,glossary,...,date_f1,date_unsupported_ratio,novel_token_ratio,copy_token_ratio,novel_content_token_ratio,s4_raw,s4_guard_fact_recall,s4_fallback_used,s5_mode,s5_num_chunks
0,0,0,S0,"(CNN)Share, and your gift will be multiplied. ...",Zully Broussard decided to give a kidney to a ...,"(CNN)Share, and your gift will be multiplied. ...",S0 Lead-3,,[],[],...,1.0,0.0,0.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN
1,0,0,S1,"(CNN)Share, and your gift will be multiplied. ...",Zully Broussard decided to give a kidney to a ...,"Here's how the super swap works, according to ...",S1 TextRank,,[],[],...,1.0,0.0,0.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN
2,0,0,S2,"(CNN)Share, and your gift will be multiplied. ...",Zully Broussard decided to give a kidney to a ...,But maybe your kidney is a match for his siste...,S2 Coverage-aware TextRank,,[],[],...,1.0,0.0,0.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN
3,0,0,S3,"(CNN)Share, and your gift will be multiplied. ...",Zully Broussard decided to give a kidney to a ...,But maybe your kidney is a match for his siste...,S3 Coverage-aware + simplification,,[],[],...,1.0,0.0,0.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN
4,1,1,S0,"(CNN)On the 6th of April 1996, San Jose Clash ...",The 20th MLS season begins this weekend .\nLea...,"(CNN)On the 6th of April 1996, San Jose Clash ...",S0 Lead-3,,[],[],...,0.2,0.0,0.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN
5,1,1,S1,"(CNN)On the 6th of April 1996, San Jose Clash ...",The 20th MLS season begins this weekend .\nLea...,"This weekend 60,000 fans are expected to witne...",S1 TextRank,,[],[],...,0.0,0.0,0.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN
6,1,1,S2,"(CNN)On the 6th of April 1996, San Jose Clash ...",The 20th MLS season begins this weekend .\nLea...,World Cup winners Kaka and David Villa will tu...,S2 Coverage-aware TextRank,,[],[],...,0.2,0.0,0.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN
7,1,1,S3,"(CNN)On the 6th of April 1996, San Jose Clash ...",The 20th MLS season begins this weekend .\nLea...,World Cup winners Kaka and David Villa will tu...,S3 Coverage-aware + simplification,,[],[],...,0.2,0.0,0.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN
8,2,2,S0,"(CNN)French striker Bafetimbi Gomis, who has a...",Bafetimbi Gomis collapses within 10 minutes of...,"(CNN)French striker Bafetimbi Gomis, who has a...",S0 Lead-3,,[],[],...,0.0,0.0,0.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN
9,2,2,S1,"(CNN)French striker Bafetimbi Gomis, who has a...",Bafetimbi Gomis collapses within 10 minutes of...,"(CNN)French striker Bafetimbi Gomis, who has a...",S1 TextRank,,[],[],...,0.0,0.0,0.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN


## 11. Aggregate comparison table

In [13]:

metric_cols = [
    "rouge1", "rouge2", "rougel",
    "flesch_reading_ease", "fkgl",
    "entity_coverage", "entity_precision", "entity_f1",
    "number_coverage", "number_precision", "number_f1",
    "date_coverage", "date_precision", "date_f1",
    "entity_unsupported_ratio", "number_unsupported_ratio", "date_unsupported_ratio",
    "copy_token_ratio", "novel_token_ratio", "novel_content_token_ratio",
    "word_count", "sentence_count", "avg_sentence_len",
]
existing_metric_cols = [c for c in metric_cols if c in combined.columns]
summary = combined.groupby("system")[existing_metric_cols].mean().round(4).reset_index()
summary_path = paths.table_dir / f"{RUN_TAG}_metric_summary.csv"
summary.to_csv(summary_path, index=False)
print("Saved summary table to:", summary_path.resolve())
summary


Saved summary table to: /Users/gm/Desktop/2/outputs/tables/validation_s0_s5_repo_integrated_v18_metric_summary.csv


,system,rouge1,rouge2,rougel,flesch_reading_ease,fkgl,entity_coverage,entity_precision,entity_f1,number_coverage,...,date_f1,entity_unsupported_ratio,number_unsupported_ratio,date_unsupported_ratio,copy_token_ratio,novel_token_ratio,novel_content_token_ratio,word_count,sentence_count,avg_sentence_len
0,S0,0.3884,0.1702,0.2468,48.5958,12.8505,0.2042,0.9993,0.3233,0.2793,...,0.4800,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,76.9136,2.9999,25.6384
1,S1,0.3343,0.1317,0.2216,50.3402,12.2529,0.1957,0.9972,0.3133,0.2109,...,0.4698,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,72.6472,2.9997,24.2162
2,S2,0.3557,0.1461,0.2304,47.9565,13.1152,0.2411,0.9999,0.3735,0.4579,...,0.7985,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,79.0337,2.9999,26.3451
3,S3,0.3556,0.1459,0.2303,48.0785,13.0982,0.2411,0.9999,0.3735,0.4579,...,0.7985,0.0000,0.0000,0.0000,0.9996,0.0004,0.0006,79.0337,2.9999,26.3451
4,S4,0.3555,0.1461,0.2325,47.3691,13.2603,0.2269,0.9922,0.3550,0.4431,...,0.7826,0.0076,0.0009,0.0001,0.9989,0.0011,0.0017,73.5709,2.7855,26.5993
5,S5,0.3961,0.1811,0.2800,55.7352,9.8728,0.1631,0.8931,0.2643,0.2879,...,0.4913,0.1060,0.0117,0.0031,0.9869,0.0131,0.0189,54.4271,3.2445,17.6608


## 12. One-example preview across all systems

In [14]:
example_id = str(df_in.iloc[0]["doc_id"])
example_rows = combined[combined["doc_id"] == example_id][["system", "output"]].copy()
example_rows


,system,output
0,S0,"(CNN)Share, and your gift will be multiplied. ..."
1,S1,"Here's how the super swap works, according to ..."
2,S2,But maybe your kidney is a match for his siste...
3,S3,But maybe your kidney is a match for his siste...
53472,S4,But maybe your kidney is a match for his siste...
66840,S5,Zully Broussard gave one of her kidneys to a s...
